In [ ]:
# Convert the original data to MNV embeddings
import os
import hashlib
from typing import List, Tuple
import torch
import numpy as np
import pandas as pd
from Bio import SeqIO
from torch import Tensor

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

input_folder = '/data/poliovirus'
output_folder = '/embeddings/poliovirus/28mnv'
os.makedirs(output_folder, exist_ok=True)

@torch.no_grad()
def calculate_counts(seq: Tensor):
    L = seq.shape[0]
    one_hot = torch.nn.functional.one_hot(seq, num_classes=4).float()
    counts = one_hot.sum(dim=0)
    positions = torch.arange(1, L + 1, device=seq.device, dtype=torch.float32)

    avg_pos = (one_hot.T @ positions) / counts.clamp(min=1e-7)

    norm_pos = positions / L * 2 * torch.pi
    cos_vals, sin_vals = torch.cos(norm_pos), torch.sin(norm_pos)
    avg_cos = (one_hot.T @ cos_vals) / counts.clamp(min=1e-7)
    avg_sin = (one_hot.T @ sin_vals) / counts.clamp(min=1e-7)

    return counts, avg_pos, avg_cos, avg_sin, one_hot, positions, norm_pos

@torch.no_grad()
def calculate_moments(
    seq: Tensor,
    counts: Tensor,
    avg_pos: Tensor,
    avg_cos: Tensor,
    avg_sin: Tensor,
    one_hot: Tensor,
    positions: Tensor,
    norm_pos: Tensor,
    k_values: List[int] = [2, 3, 4]
) -> Tuple[Tensor, Tensor, Tensor]:
    L = seq.shape[0]
    k_tensor = torch.tensor(k_values, device=seq.device, dtype=torch.float32)
    pos_diff = positions[:, None] - avg_pos[None, :]
    denominator = L * counts.clamp(min=1e-7)

    scaled_diff = pos_diff.unsqueeze(-1) / denominator[None, :, None]
    power_terms = scaled_diff ** (k_tensor - 1)
    full_terms = pos_diff.unsqueeze(-1) * power_terms
    masked_terms = full_terms * one_hot.unsqueeze(-1)
    raw_moments = masked_terms.sum(dim=0)

    cos_diff = torch.cos(norm_pos)[:, None, None] - avg_cos[None, :, None]
    sin_diff = torch.sin(norm_pos)[:, None, None] - avg_sin[None, :, None]

    cos_moments = (cos_diff ** k_tensor * one_hot.unsqueeze(-1)).sum(dim=0)
    sin_moments = (sin_diff ** k_tensor * one_hot.unsqueeze(-1)).sum(dim=0)

    cos_moments /= counts[:, None].clamp(min=1e-7)
    sin_moments /= counts[:, None].clamp(min=1e-7)

    return raw_moments, cos_moments, sin_moments

def convert(sequence: str) -> Tensor:
    mapping = {'A': 0, 'C': 1, 'G': 2, 'T': 3}
    indices = [mapping[c.upper()] if c.upper() in mapping else -1 for c in sequence]
    if -1 in indices:
        raise ValueError("Invalid nucleotide found.")
    return torch.tensor(indices, device=device, dtype=torch.long)

max_k = 2
columns = [
    'A_count', 'C_count', 'G_count', 'T_count',
    'A_avg_pos', 'C_avg_pos', 'G_avg_pos', 'T_avg_pos',
    'A_avg_pos_cos', 'C_avg_pos_cos', 'G_avg_pos_cos', 'T_avg_pos_cos',
    'A_avg_pos_sin', 'C_avg_pos_sin', 'G_avg_pos_sin', 'T_avg_pos_sin'
]

for k in range(2, max_k + 1):
    for prefix in ['', 'cos_', 'sin_']:
        columns.extend([f'{nt}_{prefix}D_{k}' for nt in ['A', 'C', 'G', 'T']])

for filename in os.listdir(input_folder):
    if not filename.endswith(".fasta"):
        continue

    print(f"Processing {filename}...")
    fasta_path = os.path.join(input_folder, filename)
    sequences, names, seen_hashes = [], [], set()

    for record in SeqIO.parse(fasta_path, "fasta"):
        try:
            seq_tensor = convert(str(record.seq))
        except ValueError:
            continue
        seq_hash = hashlib.sha256(seq_tensor.cpu().numpy().tobytes()).hexdigest()
        if seq_hash not in seen_hashes:
            seen_hashes.add(seq_hash)
            sequences.append(seq_tensor)
            names.append(record.id)

    if not sequences:
        continue

    features = []
    for seq in sequences:
        c, avg_p, avg_cos, avg_sin, one_hot, pos, norm_pos = calculate_counts(seq)
        raw_m, cos_m, sin_m = calculate_moments(seq, c, avg_p, avg_cos, avg_sin, one_hot, pos, norm_pos, list(range(2, max_k + 1)))
        feat = [
            c.cpu().numpy(), avg_p.cpu().numpy(),
            avg_cos.cpu().numpy(), avg_sin.cpu().numpy(),
            raw_m.T.flatten().cpu().numpy(),
            cos_m.T.flatten().cpu().numpy(),
            sin_m.T.flatten().cpu().numpy()
        ]
        features.append(np.concatenate(feat))

    df = pd.DataFrame(features, columns=columns)
    df.to_csv(os.path.join(output_folder, filename.replace('.fasta', '.csv')), index=False)


In [ ]:
# Calculate the convex hull disjoint ratio
import numpy as np
from scipy.optimize import linprog
import os
import pandas as pd

def intersection(mutset0, mutset00):
    m, l1 = mutset0.shape
    n, l2 = mutset00.shape
    if l1 != l2:
        print("Error: Dimensions do not match!")
        return None
    
    l = l1
    c = np.ones(m + n)
    A0 = np.hstack((mutset0.T, -mutset00.T))
    a1 = np.concatenate((np.ones(m), np.zeros(n)))
    b1 = np.concatenate((np.zeros(m), np.ones(n)))
    Aeq = np.vstack((A0, a1, b1))
    beq = np.hstack((np.zeros(l), 1, 1))
    bounds = [(0, 1)] * (m + n)
    
    # Using linprog to find the intersection
    res = linprog(c, A_eq=Aeq, b_eq=beq, bounds=bounds)
    
    if res.success:
        return 0
    else:
        return 1

# Specify the folder path
path = '/embeddings/poliovirus/28mnv'

# Initialize a list to store intersection results
results = []

# Initialize a counter for clusters with at least 3 sequences
cluster_count = 0

# Iterate over each file in the folder
files = os.listdir(path)
for i, filename1 in enumerate(files):
    if filename1.endswith('.csv'):
        file_path1 = os.path.join(path, filename1)
        
        # Read the first CSV file
        df1 = pd.read_csv(file_path1)
        
        # Check if the number of sequences is at least 3
        if len(df1) < 3:
            continue
        
        cluster_count += 1
        
        vectors1 = df1.values
        
        # Compare with other files
        for filename2 in files[i + 1:]:  # Avoid redundant comparisons
            if filename2.endswith('.csv'):
                file_path2 = os.path.join(path, filename2)
                
                # Read the second CSV file
                df2 = pd.read_csv(file_path2)
                
                # Check if the number of sequences is at least 3
                if len(df2) < 3:
                    continue
                
                vectors2 = df2.values
                
                # Check intersection
                result = intersection(vectors1, vectors2)
                
                # Store the result
                results.append(result)
            

# Calculate the ratio of successful intersections
total_intersections = sum(results)
total_comparisons = len(results)
disjoint_ratio = total_intersections / total_comparisons if total_comparisons > 0 else 0

print(f"Number of clusters with at least 3 sequences: {cluster_count}")
print(f"Disjoints: {total_intersections:.4f}")
print(f"Disjoint ratio: {disjoint_ratio:.4f}")

In [ ]:
# Neural network classification
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
import os
from sklearn.metrics import classification_report, accuracy_score
from sklearn.model_selection import StratifiedKFold

# Define neural network 
class WeightedNeuralNetwork(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, min_weight_value=1e-5):
        super(WeightedNeuralNetwork, self).__init__()
        self.input_dim = input_dim
        self.weights = nn.Parameter(torch.randn(input_dim))  
        self.min_weight_value = min_weight_value

        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_dim, hidden_dim // 2)
        self.fc3 = nn.Linear(hidden_dim // 2, output_dim)
        self.dropout = nn.Dropout(0.3)

    def forward(self, x):
        weighted_input = x * self.weights  
        out = self.fc1(weighted_input)
        out = self.relu(out)
        out = self.dropout(out)
        out = self.fc2(out)
        out = self.relu(out)
        out = self.dropout(out)
        out = self.fc3(out)
        return out

    def constraint_weights(self):
        with torch.no_grad():
            self.weights.data = torch.clamp(self.weights.data, min=self.min_weight_value)


# Check if GPU is available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Folder path
folder_path = '/embeddings/poliovirus/28mnv'

# Get all CSV file paths
csv_files = [os.path.join(folder_path, f) for f in os.listdir(folder_path) if f.endswith('.csv')]

# Initialize empty lists for storing data and labels
data_list = []
label_list = []

# Read each CSV file
for idx, file in enumerate(csv_files):
    df = pd.read_csv(file)
    # Add label column, using the file index as label
    df['label'] = idx
    data_list.append(df)

# Concatenate all data
data = pd.concat(data_list, axis=0)

# Separate features and labels
X = data.iloc[:, :-1].values  # All columns except the last one are features
y = data.iloc[:, -1].values  # The last column is the label

print(f'Number of samples after resampling: {X.shape[0]}')

# Standardize features
scaler = StandardScaler()
X = scaler.fit_transform(X)

# Convert to PyTorch tensors and move to GPU
X = torch.tensor(X, dtype=torch.float32).to(device)
y = torch.tensor(y, dtype=torch.long).to(device)

input_dim = X.shape[1]
hidden_dim = 128
output_dim = len(np.unique(y.cpu()))  

model = WeightedNeuralNetwork(input_dim, hidden_dim, output_dim).to(device)

# Loss function and optimizer
class_counts = torch.bincount(y)
class_weights = 1.0 / class_counts
loss_weights = class_weights.float().to(device)
criterion = nn.CrossEntropyLoss(weight=loss_weights)


# 5-fold cross-validation
kf = StratifiedKFold(n_splits=5, shuffle=True)

total_accuracy = 0  

for fold, (train_index, val_index) in enumerate(kf.split(X.cpu(), y.cpu())):
    print(f"\nFold {fold+1}/{5}")

    model = WeightedNeuralNetwork(input_dim, hidden_dim, output_dim).to(device)
    optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-6)

    # Split training and validation sets
    X_train, X_val = X[train_index], X[val_index]
    y_train, y_val = y[train_index], y[val_index]
    
    # Train the model
    model.train()
    for epoch in range(1000):
        optimizer.zero_grad()
        outputs = model(X_train)
        loss = criterion(outputs, y_train)
        loss.backward()
        optimizer.step()

        # Force weights to remain non-zero
        model.constraint_weights()
        
        if (epoch + 1) % 100 == 0:
            print(f'Epoch [{epoch + 1}/1000], Loss: {loss.item():.4f}')
    
    # Evaluate model on validation set
    model.eval()
    with torch.no_grad():
        outputs = model(X_val)
        _, predicted = torch.max(outputs, 1)

        y_val_cpu = y_val.cpu().numpy()
        predicted_cpu = predicted.cpu().numpy()
        
        unique_classes = np.unique(y_val_cpu)
        print("\n--- Correctly Classified Samples per Class ---")
        for cls in unique_classes:
            class_indices = np.where(y_val_cpu == cls)[0]
            correct_predictions = np.sum(predicted_cpu[class_indices] == cls)
            total_samples = len(class_indices)
            
            print(f"Class {cls}: {correct_predictions} / {total_samples}")

        accuracy = accuracy_score(y_val_cpu, predicted_cpu)
        total_accuracy += accuracy  # Accumulate accuracy
    
    print(f"Validation Accuracy for Fold {fold+1}: {accuracy:.4f}")
    print(f"Classification Report for Fold {fold+1}:\n{classification_report(y_val.cpu(), predicted.cpu(), digits=4)}")
    
# Compute and print average accuracy
average_accuracy = total_accuracy / 5
print(f"\nAverage Accuracy: {average_accuracy:.4f}")

In [ ]:
# DBSCAN clustering
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import DBSCAN
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score, fowlkes_mallows_score
from sklearn.preprocessing import LabelEncoder
import os

# load the data
folder_path = '/embeddings/poliovirus/28mnv'
files = os.listdir(folder_path)

all_data = []
all_labels = []

for file in files:
    if file.endswith('.csv'):
        file_path = os.path.join(folder_path, file)
        df = pd.read_csv(file_path)
        all_data.append(df.values)
        label = file.split('.')[0]
        all_labels.extend([label] * len(df))

data = np.vstack(all_data)

scaler = StandardScaler()
X = scaler.fit_transform(data)

y = np.array(all_labels)
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(y)

print(f"Data shape: {X.shape}, Labels shape: {y.shape}")

# clustering
dbscan = DBSCAN(eps=3.4, min_samples=5)
cluster_labels = dbscan.fit_predict(X)

# evaluation
ari = adjusted_rand_score(y, cluster_labels) if len(set(cluster_labels)) > 1 else -1
nmi = normalized_mutual_info_score(y, cluster_labels) if len(set(cluster_labels)) > 1 else -1
fmi = fowlkes_mallows_score(y, cluster_labels) if len(set(cluster_labels)) > 1 else -1

print('\nResults:')
print(f"Adjusted Rand Index: {ari:.4f}")
print(f"Normalized Mutual Information: {nmi:.4f}")
print(f"Fowlkes-Mallows Score: {fmi:.4f}")

# calculate the noise ratio
noise_ratio = np.sum(cluster_labels == -1) / len(cluster_labels)
print(f"Noise ratio: {noise_ratio:.4f}")

In [ ]:
# Check if the embeddings in the same subtype repeat
import os
import pandas as pd

# Folder path
folder_path = '/embeddings/poliovirus/28mnv'

# Get all CSV file paths
csv_files = [os.path.join(folder_path, f) for f in os.listdir(folder_path) if f.endswith('.csv')]

# Iterate through each CSV file
for file in csv_files:
    # Read CSV file
    df = pd.read_csv(file)
    
    # Check for duplicate rows
    duplicates = df[df.duplicated(keep=False)]  # Keep all duplicate rows, including the first occurrence
    
    if len(duplicates) > 0:
        print(f"Duplicate rows found in file {file}:")
        
        # Find duplicate rows by grouping on all columns
        duplicate_groups = duplicates.groupby(list(duplicates.columns)).size()
        
        # Iterate through each duplicate group
        for index, count in duplicate_groups.items():
            if count > 1:
                print(f"Group {index} repeated {count} times")
    else:
        print(f"No duplicate rows found in file {file}")

In [ ]:
# Shuffle the labels to validate the convex hull results

import numpy as np
from scipy.optimize import linprog
import os
import pandas as pd
import random
import time

# Set maximum number of comparisons to avoid computing all combinations
MAX_COMPARISONS = 2000  # Adjust as needed

def intersection(mutset0, mutset00):
    m, l1 = mutset0.shape
    n, l2 = mutset00.shape
    if l1 != l2:
        print("Error: Dimensions do not match!")
        return None
    
    l = l1
    c = np.ones(m + n)
    A0 = np.hstack((mutset0.T, -mutset00.T))
    a1 = np.concatenate((np.ones(m), np.zeros(n)))
    b1 = np.concatenate((np.zeros(m), np.ones(n)))
    Aeq = np.vstack((A0, a1, b1))
    beq = np.hstack((np.zeros(l), 1, 1))
    bounds = [(0, 1)] * (m + n)
    
    try:
        res = linprog(c, A_eq=Aeq, b_eq=beq, bounds=bounds, method='highs')
    except:
        res = linprog(c, A_eq=Aeq, b_eq=beq, bounds=bounds)
    
    if res.success:
        return 0  # Convex hulls intersect
    else:
        return 1  # Convex hulls disjoint

# Specify folder path
path = '/embeddings/poliovirus/28mnv'  

# Store all vectors and original cluster labels
all_vectors = []
cluster_labels = []
valid_clusters = []

print("Reading data and identifying valid clusters...")
start_time = time.time()

# First pass: collect all data and identify valid clusters
files = os.listdir(path)
for filename in files:
    if filename.endswith('.csv'):
        file_path = os.path.join(path, filename)
        df = pd.read_csv(file_path)
        
        if len(df) >= 3: 
            vectors = df.values
            all_vectors.extend(vectors)
            cluster_labels.extend([filename] * len(vectors))
            valid_clusters.append((filename, vectors))

print(f"Number of valid clusters: {len(valid_clusters)}")
print(f"Data reading completed, time taken: {time.time() - start_time:.2f} seconds")


# Original analysis (using true labels)
original_results = []
total_pairs = len(valid_clusters) * (len(valid_clusters) - 1) // 2

# Decide whether to sample
if total_pairs > MAX_COMPARISONS:
    print(f"Large number of pairs ({total_pairs}), randomly sampling {MAX_COMPARISONS} pairs for estimation.")
    # Randomly sample MAX_COMPARISONS pairs
    indices = list(range(len(valid_clusters)))
    pairs_to_compare = random.sample([(i, j) for i in range(len(valid_clusters)) for j in range(i+1, len(valid_clusters))], MAX_COMPARISONS)
    for idx, (i, j) in enumerate(pairs_to_compare):
        vecs1 = valid_clusters[i][1]
        vecs2 = valid_clusters[j][1]
        result = intersection(vecs1, vecs2)
        original_results.append(result)
        if (idx + 1) % 100 == 0:
            print(f"Original analysis progress: {idx + 1}/{MAX_COMPARISONS}")
else:
    print(f"Total pairs: {total_pairs}, computing all pairs.")
    for i, (name1, vecs1) in enumerate(valid_clusters):
        for j, (name2, vecs2) in enumerate(valid_clusters[i+1:], i+1):
            result = intersection(vecs1, vecs2)
            original_results.append(result)
            if len(original_results) % 100 == 0:
                print(f"Original analysis progress: {len(original_results)}/{total_pairs}")

total_comparisons = len(original_results)
original_disjoints = sum(original_results)
original_disjoint_ratio = original_disjoints / total_comparisons if total_comparisons > 0 else 0

print("\n=== Original Analysis (True Labels) ===")
print(f"Total comparisons: {total_comparisons}")
print(f"Number of disjoint pairs: {original_disjoints}")
print(f"Disjoint ratio: {original_disjoint_ratio:.4f}")

# Random label experiment
num_trials = 10  # Number of randomization trials
random_ratios = []

print("\nStarting random label experiment...")
random_start_time = time.time()

for trial in range(num_trials):
    trial_start_time = time.time()
    # Create shuffled labels
    shuffled_labels = cluster_labels.copy()
    random.shuffle(shuffled_labels)
    
    # Regroup vectors based on shuffled labels
    shuffled_clusters = {}
    for vector, label in zip(all_vectors, shuffled_labels):
        if label not in shuffled_clusters:
            shuffled_clusters[label] = []
        shuffled_clusters[label].append(vector)
    
    # Filter out clusters with ≥3 sequences
    valid_shuffled = []
    for label, vectors in shuffled_clusters.items():
        if len(vectors) >= 3:
            valid_shuffled.append((label, np.array(vectors)))
    
    # Calculate number of valid clusters for current trial
    k = len(valid_shuffled)
    total_pairs_shuffled = k * (k - 1) // 2
    
    trial_disjoints = 0
    trial_comparisons = 0
    
    if total_pairs_shuffled > MAX_COMPARISONS:
        # Randomly sample MAX_COMPARISONS pairs
        pairs_to_compare = random.sample([(i, j) for i in range(k) for j in range(i+1, k)], MAX_COMPARISONS)
        for idx, (i, j) in enumerate(pairs_to_compare):
            vecs1 = valid_shuffled[i][1]
            vecs2 = valid_shuffled[j][1]
            result = intersection(vecs1, vecs2)
            trial_disjoints += result
            trial_comparisons += 1
            if (idx + 1) % 100 == 0:
                print(f"Trial {trial+1} progress: {idx + 1}/{MAX_COMPARISONS}")
    else:
        for i, (_, vecs1) in enumerate(valid_shuffled):
            for j, (_, vecs2) in enumerate(valid_shuffled[i+1:], i+1):
                result = intersection(vecs1, vecs2)
                trial_disjoints += result
                trial_comparisons += 1
                if trial_comparisons % 100 == 0:
                    print(f"Trial {trial+1} progress: {trial_comparisons}/{total_pairs_shuffled}")
    
    if trial_comparisons > 0:
        disjoint_ratio = trial_disjoints / trial_comparisons
        random_ratios.append(disjoint_ratio)
    
    trial_time = time.time() - trial_start_time
    print(f"Completed trial {trial+1}/{num_trials}, time taken: {trial_time:.2f} seconds")

random_time = time.time() - random_start_time
print(f"Random experiment total time: {random_time:.2f} seconds")

# Calculate statistics for random experiment
avg_random_ratio = np.mean(random_ratios) if random_ratios else 0
std_random_ratio = np.std(random_ratios) if random_ratios else 0

print("\n=== Random Label Analysis ===")
print(f"Number of trials: {num_trials}")
print(f"Average disjoint ratio: {avg_random_ratio:.4f}")
print(f"Standard deviation: {std_random_ratio:.4f}")

# Compare original and random results
print("\n=== Significance Analysis ===")
print(f"Original disjoint ratio: {original_disjoint_ratio:.4f}")
print(f"Random disjoint ratio: {avg_random_ratio:.4f} ± {std_random_ratio:.4f}")

if original_disjoint_ratio > avg_random_ratio + 2 * std_random_ratio:
    print("Result: Original disjoint ratio is significantly higher than random case")
    print("Conclusion: Convex hull separation has classification significance")
else:
    print("Result: No significant difference between original and random cases")
    print("Conclusion: Convex hull separation may not indicate true category structure")